In [4]:
import os

checkpoints_root = "/kaggle/input/datasets/ahsanulhoque99/gnn-bert-fusion-dataset"
print(os.listdir(checkpoints_root))

['task3_fusion.pt']


In [2]:
!pip install -q torch_geometric

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
from torch_geometric.nn import SAGEConv, global_mean_pool
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

mtat_root = "/kaggle/input/datasets/yalumusic/magnatagatune"
audio_dir = f"{mtat_root}/MagnaTagATune"

tags_df = pd.read_csv(f"{mtat_root}/annotations_final.csv", sep="\t")
tag_columns = tags_df.columns[1:-1]
top_50_tags = tags_df[tag_columns].sum().sort_values(ascending=False).head(50).index.tolist()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.0 MB/s eta 0:00:00a 0:00:01
Device: cuda


In [5]:
def load_audio(path, sr=22050):
    y, sr = librosa.load(path, sr=sr, mono=True)
    return y, sr

def get_melspectrogram(y, sr, n_mels=128):
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    log_mel = librosa.power_to_db(mel, ref=1.0).T
    mean = log_mel.mean(axis=0, keepdims=True)
    std = log_mel.std(axis=0, keepdims=True) + 1e-8
    return (log_mel - mean) / std

def split_into_segments(feature_matrix, sr, hop_length=512, window_seconds=5.0):
    frames_per_segment = int(window_seconds * sr / hop_length)
    segments = []
    for start in range(0, feature_matrix.shape[0], frames_per_segment):
        chunk = feature_matrix[start:start + frames_per_segment]
        if len(chunk) > 0:
            segments.append(chunk.mean(axis=0))
    return segments

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

def build_segment_graph(segments, similarity_threshold=0.5):
    n = len(segments)
    edges = set()
    for i in range(n - 1):
        edges.add((i, i + 1))
    for i in range(n):
        for j in range(i + 1, n):
            if cosine_similarity(segments[i], segments[j]) > similarity_threshold:
                edges.add((i, j))
    return list(edges)


class GNNEncoder(nn.Module):
    def __init__(self, in_dim=128, hidden_dim=128, num_layers=3, dropout=0.2):
        super().__init__()
        self.convs = nn.ModuleList()
        dims = [in_dim] + [hidden_dim] * num_layers
        for i in range(num_layers):
            self.convs.append(SAGEConv(dims[i], dims[i + 1]))
        self.dropout = dropout
        self.out_dim = hidden_dim
    def forward(self, x, edge_index, batch):
        h = x
        for i, conv in enumerate(self.convs):
            h = conv(h, edge_index)
            if i < len(self.convs) - 1:
                h = torch.relu(h)
                h = nn.functional.dropout(h, p=self.dropout, training=self.training)
        return global_mean_pool(h, batch)


class FusionModel(nn.Module):
    def __init__(self, gnn_encoder, bert_model, num_tags=50, fused_dim=128):
        super().__init__()
        self.gnn = gnn_encoder
        self.bert = bert_model
        self.graph_proj = nn.Linear(gnn_encoder.out_dim, fused_dim)
        self.text_proj = nn.Linear(bert_model.config.hidden_size, fused_dim)
        self.attention = nn.MultiheadAttention(embed_dim=fused_dim, num_heads=4, batch_first=True)
        self.tag_head = nn.Linear(fused_dim * 2, num_tags)
        self.emotion_head = nn.Linear(fused_dim * 2, 2)
    def forward(self, graph_x, graph_edge_index, graph_batch, input_ids, attention_mask):
        graph_vector = self.gnn(graph_x, graph_edge_index, graph_batch)
        text_tokens = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        g = self.graph_proj(graph_vector).unsqueeze(1)
        t = self.text_proj(text_tokens)
        attended, _ = self.attention(query=g, key=t, value=t)
        combined = torch.cat([g.squeeze(1), attended.squeeze(1)], dim=-1)
        return self.tag_head(combined), self.emotion_head(combined)

In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased")

gnn_encoder = GNNEncoder(in_dim=128, hidden_dim=128, num_layers=3)
fusion_model = FusionModel(gnn_encoder, bert, num_tags=50).to(device)
fusion_model.load_state_dict(torch.load(f"{checkpoints_root}/task3_fusion.pt", map_location=device))
fusion_model.eval()
print("Fusion model loaded from checkpoint successfully.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fusion model loaded from checkpoint successfully.


In [7]:
mtat_df = tags_df[tags_df[top_50_tags].sum(axis=1) > 0].reset_index(drop=True)
print("mtat_df rows:", len(mtat_df))  # should print 21111

mtat_df rows: 21111


# **rebuild the test split**

In [8]:
train_folders = list("0123456789ab")
val_folders = ["c"]
test_folders = ["d", "e", "f"]

mtat_df["folder"] = mtat_df["mp3_path"].str.split("/").str[0]
test_df = mtat_df[mtat_df["folder"].isin(test_folders)].reset_index(drop=True)

failed_paths = [
    "6/norine_braun-now_and_zen-08-gently-117-146.mp3",
    "8/jacob_heringman-josquin_des_prez_lute_settings-19-gintzler__pater_noster-204-233.mp3",
    "9/american_baroque-dances_and_suites_of_rameau_and_couperin-25-le_petit_rien_xiveme_ordre_couperin-88-117.mp3",
]
test_df = test_df[~test_df["mp3_path"].isin(failed_paths)].reset_index(drop=True)
print("Test set size:", len(test_df))  # should print 4332

Test set size: 4332


# **cache the test clips' graphs**

In [9]:
import os, time
from torch_geometric.data import Data

os.makedirs("data/processed", exist_ok=True)

def process_and_cache_clip(mp3_path, save_path):
    y, sr = load_audio(f"{audio_dir}/{mp3_path}")
    mel = get_melspectrogram(y, sr)
    segments = split_into_segments(mel, sr)
    edges = build_segment_graph(segments)
    x = torch.tensor(np.array(segments), dtype=torch.float32)
    edge_index = torch.tensor([[0], [0]], dtype=torch.long) if len(edges) == 0 else torch.tensor(edges, dtype=torch.long).t().contiguous()
    torch.save(Data(x=x, edge_index=edge_index), save_path)

start = time.time()
processed, failed = 0, 0
for i, row in test_df.iterrows():
    save_path = f"data/processed/{row['clip_id']}.pt"
    if os.path.exists(save_path):
        continue
    try:
        process_and_cache_clip(row["mp3_path"], save_path)
        processed += 1
    except Exception as e:
        failed += 1
    if (i + 1) % 1000 == 0:
        print(f"{i+1}/{len(test_df)} | {(time.time()-start)/60:.1f} min")

print(f"Done. Processed {processed}, failed {failed}. Total: {(time.time()-start)/60:.1f} min")

1000/4332 | 1.8 min
2000/4332 | 3.3 min
3000/4332 | 4.9 min
4000/4332 | 6.4 min
Done. Processed 4332, failed 0. Total: 7.0 min


# **FusionTagDatasetV2**

In [46]:
from torch.utils.data import Dataset

class FusionTagDatasetV2(Dataset):
    def __init__(self, df, tag_columns, tokenizer, max_length=32, mask_ratio=0.5, seed=42):
        self.df = df.reset_index(drop=True)
        self.tag_columns = tag_columns
        self.tokenizer = tokenizer
        self.max_length = max_length

        rng = np.random.RandomState(seed)
        self.captions = []
        for i in range(len(self.df)):
            row = self.df.iloc[i]
            true_tags = [tag for tag in tag_columns if row[tag] == 1]
            if not true_tags:
                self.captions.append("[no tags]")
                continue
            n_keep = max(1, round(len(true_tags) * (1 - mask_ratio)))
            kept = rng.choice(true_tags, size=n_keep, replace=False)
            self.captions.append(" ".join(kept))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        graph = torch.load(f"data/processed/{row['clip_id']}.pt", weights_only=False)
        tokens = self.tokenizer(
            self.captions[idx], padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt",
        )
        graph.input_ids = tokens["input_ids"]
        graph.attention_mask = tokens["attention_mask"]
        graph.y = torch.tensor(row[self.tag_columns].values.astype("float32")).unsqueeze(0)
        return graph

test_fusion_dataset = FusionTagDatasetV2(test_df, top_50_tags, tokenizer)
print("test_fusion_dataset built:", len(test_fusion_dataset), "clips")

test_fusion_dataset built: 4332 clips


# **run inference on real clip**

In [51]:
def predict_tags(mp3_path, shown_tag_words):
    y, sr = load_audio(mp3_path)
    mel = get_melspectrogram(y, sr)
    segments = split_into_segments(mel, sr)
    edges = build_segment_graph(segments)

    x = torch.tensor(np.array(segments), dtype=torch.float32).to(device)
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous().to(device) if edges else torch.tensor([[0], [0]]).to(device)
    batch_vec = torch.zeros(x.size(0), dtype=torch.long, device=device)

    caption = " ".join(shown_tag_words) if shown_tag_words else "[no tags]"
    tokens = tokenizer(caption, padding="max_length", truncation=True, max_length=32, return_tensors="pt").to(device)

    with torch.no_grad():
        tag_logits, _ = fusion_model(x, edge_index, batch_vec, tokens["input_ids"], tokens["attention_mask"])
        probs = torch.sigmoid(tag_logits)[0]

    predicted = [top_50_tags[i] for i in range(50) if probs[i] > 0.5]
    return caption, predicted



rng_seed = None
candidates = tags_df[tags_df[top_50_tags].sum(axis=1) >= 3].reset_index(drop=True)
sample_row = candidates.sample(1, random_state=rng_seed).iloc[0] if rng_seed is not None else candidates.sample(1).iloc[0]

sample_path = f"{audio_dir}/{sample_row['mp3_path']}"
true_tags = [t for t in top_50_tags if sample_row[t] == 1]
shown = list(np.random.RandomState(1).choice(true_tags, size=max(1, len(true_tags)//2), replace=False))

caption_used, predicted_tags = predict_tags(sample_path, shown)

print("=== End-to-end inference example ===")
print("Audio file:", sample_row["mp3_path"])
print("Partial tags shown to model:", caption_used)
print("True full tag set:", true_tags)
print("Model's predicted tag set:", predicted_tags)

=== End-to-end inference example ===
Audio file: 3/dj_cary-downtempo_chill-13-oui_mossieur_chuibelle_dr_kuch-175-204.mp3
Partial tags shown to model: man beat
True full tag set: ['ambient', 'beat', 'man', 'voice', 'beats']
Model's predicted tag set: ['beat', 'man']


In [52]:
def evaluate_held_out_recovery_full(dataset, model, device, top_50_tags):
    """
    Uses the SAME captions/masking already built into test_fusion_dataset
    (seed=42, mask_ratio=0.5 -- identical to official evaluation), so this
    number is directly comparable to the reported test macro-F1, not a
    separately-sampled estimate.
    """
    model.eval()
    total_held_out, total_recovered = 0, 0
    total_shown_correct, total_shown = 0, 0

    with torch.no_grad():
        for i in range(len(dataset)):
            graph = dataset[i]
            caption = dataset.captions[i]
            shown_words = set(caption.split()) if caption != "[no tags]" else set()

            true_tags = [t for j, t in enumerate(top_50_tags) if graph.y[0][j] == 1]
            held_out = [t for t in true_tags if t not in shown_words]
            shown = [t for t in true_tags if t in shown_words]

            batch_vec = torch.zeros(graph.x.size(0), dtype=torch.long, device=device)
            tag_logits, _ = model(
                graph.x.to(device), graph.edge_index.to(device), batch_vec,
                graph.input_ids.to(device), graph.attention_mask.to(device)
            )
            probs = torch.sigmoid(tag_logits)[0]
            predicted = set(top_50_tags[j] for j in range(50) if probs[j] > 0.5)

            total_held_out += len(held_out)
            total_recovered += sum(1 for t in held_out if t in predicted)
            total_shown += len(shown)
            total_shown_correct += sum(1 for t in shown if t in predicted)

    print(f"Held-out tags recovered: {total_recovered}/{total_held_out} ({100*total_recovered/total_held_out:.1f}%)")
    print(f"Shown tags correctly echoed: {total_shown_correct}/{total_shown} ({100*total_shown_correct/total_shown:.1f}%)")

evaluate_held_out_recovery_full(test_fusion_dataset, fusion_model, device, top_50_tags)

Held-out tags recovered: 1930/6868 (28.1%)
Shown tags correctly echoed: 7059/7281 (97.0%)


In [49]:
import os
os.makedirs("results", exist_ok=True)



import json
held_out_analysis = {
    "held_out_recovered": 1930, "held_out_total": 6868, "held_out_recovery_rate": 1930/6868,
    "shown_correctly_echoed": 7059, "shown_total": 7281, "echo_rate": 7059/7281,
}
with open("results/held_out_recovery_analysis.json", "w") as f:
    json.dump(held_out_analysis, f, indent=2)
print("Saved.")

Saved.


In [50]:
mtat_results = pd.DataFrame({
    "B1 (majority-class)":             {"macro_f1": 0.0000, "pr_auc": 0.0653},
    "B2 (CNN-mel)":                     {"macro_f1": 0.2152, "pr_auc": 0.1842},
    "Task 2: GNN-only":                 {"macro_f1": 0.2649, "pr_auc": 0.2437},
    "Task 3: Fusion (early-concat)":    {"macro_f1": 0.7196, "pr_auc": 0.7776},
    "Task 3: Fusion (cross-attention)": {"macro_f1": 0.7096, "pr_auc": 0.7628},
    "Task 1: BERT-only":                {"macro_f1": 0.7272, "pr_auc": 0.7816},
}).T.sort_values("macro_f1")

print("=== MagnaTagATune: all models ===")
print(mtat_results)

print("\n=== GTZAN genre classification ===")
print(pd.DataFrame({"CNN": {"accuracy": 0.34, "macro_f1": 0.3088}, "GNN": {"accuracy": 0.37, "macro_f1": 0.3765}}).T)

print("\n=== Task 4: contrastive retrieval (test set) ===")
print(pd.DataFrame({
    "Caption->Audio": {"R@1": 0.0050, "R@5": 0.0301, "R@10": 0.0565},
    "Audio->Caption": {"R@1": 0.0063, "R@5": 0.0251, "R@10": 0.0452},
}).T)

=== MagnaTagATune: all models ===
                                  macro_f1  pr_auc
B1 (majority-class)                 0.0000  0.0653
B2 (CNN-mel)                        0.2152  0.1842
Task 2: GNN-only                    0.2649  0.2437
Task 3: Fusion (cross-attention)    0.7096  0.7628
Task 3: Fusion (early-concat)       0.7196  0.7776
Task 1: BERT-only                   0.7272  0.7816

=== GTZAN genre classification ===
     accuracy  macro_f1
CNN      0.34    0.3088
GNN      0.37    0.3765

=== Task 4: contrastive retrieval (test set) ===
                   R@1     R@5    R@10
Caption->Audio  0.0050  0.0301  0.0565
Audio->Caption  0.0063  0.0251  0.0452
